<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
</style>

# All Five Sensors

The downward ToF sensor provides a distance observation toward the surface beneath the Duckiedrone, contributing to height estimation when interpreted with the sensor's orientation and the scene geometry. The other four sensors extend that view upward, forward, left, and right, allowing a program to observe nearby surfaces in several directions.

[Subscribing to the ToF sensor data](./4-subscribing-to-tof.ipynb) stored and displayed readings from one topic. This notebook extends that subscriber to all five sensors and investigates how their independently arriving measurements appear together in one printed line.

This notebook addresses the following questions:

- Which services supply the four additional sensor streams?

- How does one subscriber associate each message with its sensor?

- What does a line containing five ranges reveal about the surrounding scene?

- How can the timing of those readings affect their interpretation?

## The five sensors

[Table 1](#table-1) summarizes the default configuration introduced in [Duckiedrone ToF Sensor Architecture](./3-duckiedrone-tof-architecture.ipynb#table-3).

<table id="table-1" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 1: Sensor directions, requested frequencies, and default startup state.</caption>
  <thead>
    <tr><th>Sensor</th><th>Viewing direction</th><th>Requested frequency</th><th>Started by default</th></tr>
  </thead>
  <tbody>
    <tr><td><code>bottom</code></td><td>Downward</td><td>30 Hz</td><td>Yes</td></tr>
    <tr><td><code>front</code></td><td>Forward</td><td>5 Hz</td><td>No</td></tr>
    <tr><td><code>left</code></td><td>Left</td><td>5 Hz</td><td>No</td></tr>
    <tr><td><code>right</code></td><td>Right</td><td>5 Hz</td><td>No</td></tr>
    <tr><td><code>top</code></td><td>Upward</td><td>5 Hz</td><td>No</td></tr>
  </tbody>
</table>

Each sensor uses its own driver configuration and ROS 2 bridge, allowing its acquisition settings to be adjusted separately. Starting the additional four sensors therefore adds eight containers to the Duckiedrone's Docker stack.

## Activity 1: start the other four sensors

<a id="activity-1"></a>

Begin with the working bottom-sensor stream established in [Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb). Keep a physical Duckiedrone disarmed throughout the activity. For a virtual Duckiedrone, keep the Duckiematrix running with its entity attached.

In the Duckiedrone device shell, inspect the current ToF containers:

```bash
docker ps --format '{{.Names}}' | grep tof
```

Record which sensor services are already running so that the original state can be restored afterward. In the default setup, the list contains `driver-tof-bottom` and `ros2-tof-bottom`.

From a separate base-station terminal, start the additional services:

```bash
dts stack up -H DUCKIEDRONE_NAME -d duckiedrone/extra_sensors
```

The command starts the front, left, right, and top driver/bridge pairs. The `-d` option leaves them running after the command returns.

Repeat the container listing in the device shell. With all five paths running, it should contain `driver-tof-<sensor>` and `ros2-tof-<sensor>` for each name in Table 1.

Open a shell inside the bottom bridge, or reuse the bridge shell from the preceding notebook:

```bash
docker exec -it ros2-tof-bottom bash
```

Then list the ToF topics:

```bash
ros2 topic list | grep tof
```

Look for these five names, with the actual Duckiedrone namespace:

```text
/DUCKIEDRONE_NAME/bottom_tof_driver_node/range
/DUCKIEDRONE_NAME/front_tof_driver_node/range
/DUCKIEDRONE_NAME/left_tof_driver_node/range
/DUCKIEDRONE_NAME/right_tof_driver_node/range
/DUCKIEDRONE_NAME/top_tof_driver_node/range
```

If a name is missing, check its driver/bridge pair before proceeding.

Compare arrivals from the bottom sensor and one newly started sensor. Run the following commands separately, allowing several reporting windows and stopping each with `Ctrl+C`:

```bash
ros2 topic hz /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

```bash
ros2 topic hz /DUCKIEDRONE_NAME/front_tof_driver_node/range
```

Record the two observed rates and compare them with Table 1. A waiting command indicates that it has yet to receive enough messages to report a rate. Use the publisher and container checks in [Subscribing to the ToF sensor data](./4-subscribing-to-tof.ipynb) to investigate.


## Activity 2: subscribe to all five

<a id="activity-2"></a>

The subscriber takes a list of sensor names and constructs one topic for each name. The common naming pattern makes the same code usable for one sensor or several.

Open [tof_math.py](../packages/solution/solution/tof_math.py) in the LX editor. In the learner starter, complete `topics_for` at its `TO IMPLEMENT` block. In the solution version, inspect the existing implementation.

For example, the input:

```python
["bottom", "front"]
```

should produce:

```python
[
    "/bottom_tof_driver_node/range",
    "/front_tof_driver_node/range",
]
```

Preserve the input order. `ToFSubscriber` pairs each returned topic with the corresponding sensor name and adds the Duckiedrone namespace.

### Tell the launch file which sensors to use

Open [tof_listener_node.launch.py](../packages/tof_listener/launch/tof_listener_node.launch.py) and locate:

```python
sensor_names_arg = DeclareLaunchArgument(
    "sensor_names",
    default_value="['bottom']",
    description="Names of the ToF sensors to subscribe to.",
)
```

Change its default list to include all five:

```python
    default_value="['bottom','front','left','right','top']",
```

The launch argument supplies the names to `ToFSubscriber`, which passes them to `topics_for` and creates the subscriptions. Each callback retains its associated sensor name.

### Deploy it

Stop the previous workbench run with `Ctrl+C` and save the edits. From the LX root in the base-station terminal, build:

```bash
dts code build -R DUCKIEDRONE_NAME
```

After the build succeeds, start the workbench:

```bash
dts code workbench -R DUCKIEDRONE_NAME
```

The startup output should name five subscription topics. Compare them with the topics inspected in Activity 1.

As messages arrive, their sensor names appear in the periodic summaries. An illustrative pair of lines is:

```text
ranges=[bottom=0.013m front=0.116m left=1.288m right=0.483m top=1.660m]
ranges=[bottom=0.012m front=0.116m left=1.309m right=0.482m top=1.663m]
```

The logger sorts entries alphabetically by sensor name. Each entry first appears after that sensor's callback receives a message, so the initial summaries can contain only part of the configured list.

If an entry remains absent, inspect that sensor's topic, publisher, and incoming messages. Once an entry has appeared, the dictionary retains it during subsequent gaps.

The example bottom ranges are below the advertised minimum. Apply the range-limit and freshness checks developed in [Subscribing to the ToF sensor data](./4-subscribing-to-tof.ipynb) when interpreting any entry.


### Relate the readings to the scene

Keep the physical Duckiedrone stationary with its motors stopped. Place a broad, flat target in front of one sideways sensor, initially about 20 to 30 cm away, and move it slowly.

Observe which entry responds. Repeat for another direction, considering which surfaces lie within each sensor's field of view. The once-per-second display makes slow movements easier to follow.

On a virtual Duckiedrone, relate the entries to the surrounding simulated surfaces as the entity's position changes.

Use the single-topic inspection command when a displayed value is difficult to explain:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/front_tof_driver_node/range
```

This prints the full message, including its timestamp and range bounds, then exits after reception.

### Five readings, separate update times

Each callback updates one dictionary entry when its message arrives. At the timer event, the logger combines the entries currently stored. Their timestamps can therefore differ.

The default requested frequencies correspond to periods of approximately 33 ms for the bottom sensor and 200 ms for the other four. Actual reading ages also depend on acquisition, communication, scheduling, and gaps in delivery. During an interruption, the stored value continues aging until another message replaces it.

Timing differences become important when the scene or Duckiedrone moves. At 1 m/s, a displacement over 0.2 s is 0.2 m. Combining ranges collected across such an interval requires accounting for the motion.

The subscriber already retains `stamp_sec` for each reading. Printing those timestamps alongside the ranges makes the separate updates visible, including cases where a timestamp advances while the rounded distance stays the same.


## Going further

The following extensions build on the five-sensor subscriber.

### Select a subset

Set the launch argument to:

```python
    default_value="['bottom','front']",
```

Save, rebuild, and restart the workbench. Compare the two startup topics and resulting entries with the five-sensor run.

The launch list controls the listener's subscriptions. The driver and bridge services continue according to their own running state.

### Inspect timing

Add `stamp_sec` to the logger's output and compare successive summaries.

To calculate age from a source timestamp, first establish that the source and subscriber clocks use a compatible time basis. Clock offsets and simulated time can otherwise affect the subtraction.

For an observation based entirely on the receiving process, record local monotonic time in each callback and display elapsed time since that sensor's last arrival. This measures how recently the subscriber received an update.

### Observe a stream directly

In the bridge shell, run:

```bash
ros2 topic echo /DUCKIEDRONE_NAME/front_tof_driver_node/range --field range
```

Move the target slowly and compare the arriving values with the workbench summaries. Stop the command with `Ctrl+C`.

### Change a physical sensor's requested rate

The left sensor provides a configuration experiment without changing the bottom-sensor path used by the flight controller.

In the Duckiedrone device shell, outside the driver and ROS 2 bridge containers, save the current left-sensor configuration before editing it:

```bash
cp /data/config/node/tof_driver_left/DUCKIEDRONE_NAME.yaml \
  /tmp/tof_driver_left_before_rate_experiment.yaml
```

First measure its arrival rate with `ros2 topic hz` in the bridge shell, then stop that command with `Ctrl+C`. Return to the device shell and edit the configuration with:

```bash
sudo nano /data/config/node/tof_driver_left/DUCKIEDRONE_NAME.yaml
```

A 10 Hz target is compatible with:

```yaml
frequency: 10
timing_budget: 100
```

If `inter_measurement_period` is explicitly present, set it to 100 ms for this experiment. When omitted, the driver derives the period from the frequency and timing budget.

Save the file (`Ctrl-X`, `Y`, `Enter`), then restart the left driver followed by its bridge:

```bash
docker restart driver-tof-left
docker restart ros2-tof-left
```

In a bridge shell, observe the resulting stream:

```bash
ros2 topic hz /DUCKIEDRONE_NAME/left_tof_driver_node/range
```

Allow several reporting windows, record the result, and stop with `Ctrl+C`. Compare it with the earlier observation and the configured timing constraints.

Afterward, restore the saved configuration from the device shell:

```bash
sudo cp /tmp/tof_driver_left_before_rate_experiment.yaml \
  /data/config/node/tof_driver_left/DUCKIEDRONE_NAME.yaml
```

Repeat the restart sequence and check message arrivals again. Physical acquisition settings should be evaluated using the physical sensor path.

## Restore the starting configuration

Stop the workbench run with `Ctrl+C`.

If this activity started the extra-sensors stack from its default stopped state, stop it from the base-station terminal:

```bash
dts stack down -H DUCKIEDRONE_NAME duckiedrone/extra_sensors
```

If the extra sensors were already running before the activity, retain their original running state.

In the device shell, repeat:

```bash
docker ps --format '{{.Names}}' | grep tof
```

After stopping the extra stack, the bottom driver/bridge pair should remain. Confirm that its stream still arrives from the bottom bridge shell:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

Restore any experimental driver settings and the launch sensor list to their pre-activity values. Save the source edits and rebuild before the next workbench run.


## Summary and next steps

The five sensors supply distance observations in different directions through separate driver/bridge pairs. A list of sensor names lets the subscriber construct their topics and retain each reading under the correct name.

The combined display brings those observations together for inspection. Each entry carries its own update history, so interpretation combines scene geometry, range validity, and timing.

This completes the ToF sensor LX. The same questions provide a starting point for the camera LX: what the sensor observes, how the observation reaches a program, and which conditions affect its usefulness.

## Further reading

- [Extra-sensors stack definition](https://github.com/duckietown/duckietown-shell-commands/blob/ente/stack/stacks/duckiedrone/extra_sensors.yaml): the four additional driver/bridge pairs.

- [Range message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg): fields and range limits.

- [Duckiedrone ToF Sensor Architecture](./3-duckiedrone-tof-architecture.ipynb#changing-any-of-this): timing constraints and applying driver settings.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()